# Generación de SBOMs

## Objetivo

Este notebook explica el proceso para generar **Software Bill of Materials (SBOMs)** para un conjunto de repositorios de código y cómo analizar los resultados de manera centralizada. Utilizaremos la herramienta **Syft** para la generación y la librería **Pandas** en Python para el análisis.

El flujo de trabajo completo está diseñado para ser reproducible y se basa en las siguientes etapas:

1.  **Descubrimiento de Repositorios**: Identificar los proyectos a los que se les generará un SBOM.
2.  **Generación de SBOMs**: Ejecutar Syft para analizar cada repositorio y crear un SBOM en formato JSON.
3.  **Análisis de Resultados**: Cargar todos los SBOMs generados en un DataFrame de Pandas para su inspección y análisis.


## Estructura de Directorios

El proyecto sigue una estructura organizada para separar los datos de entrada, los scripts y los resultados:

```
ciberseguridad_2026/
├── data/
│   ├── repos/      # Directorio para los repositorios a analizar
│   └── results/    # Directorio para los SBOMs generados en JSON
├── nbs/            # Notebooks de Jupyter
└── scripts/        # Scripts de automatización
    └── generate_sboms.py
```

## Paso 1: Generación de SBOMs

El primer paso es ejecutar el script `generate_sboms.py`, que se encarga de orquestar la generación de los SBOMs.

Este script realiza las siguientes acciones:

1.  **Busca repositorios**: Escanea el directorio `data/repos` en busca de subdirectorios que contengan código fuente.
2.  **Ejecuta Syft**: Para cada repositorio encontrado, invoca a `syft` para analizar su contenido.
3.  **Guarda los resultados**: El SBOM generado para cada repositorio se guarda como un archivo JSON en el directorio `data/results`.

In [1]:
!python ../../scripts/generate_sboms.py

INFO | Usando Syft CLI: /home/hector/An-lisis-de-Vulnerabilidades/.venv/bin/syft
INFO | [1/5] Procesando repositorio data/repos/azure-sdk-for-python
INFO | SBOM guardado en data/results/azure-sdk-for-python.json
INFO | [2/5] Procesando repositorio data/repos/debugpy
INFO | SBOM guardado en data/results/debugpy.json
INFO | [3/5] Procesando repositorio data/repos/java
INFO | SBOM guardado en data/results/java.json
INFO | [4/5] Procesando repositorio data/repos/pyright
INFO | SBOM guardado en data/results/pyright.json
INFO | [5/5] Procesando repositorio data/repos/semantic-kernel
INFO | SBOM guardado en data/results/semantic-kernel.json
INFO | Resumen final | total_repos=5 | repos_generados=5 | archivos_generados=5 | omitidos=0 | errores=0


## Paso 2: Análisis de los SBOMs con Pandas

Una vez que los SBOMs han sido generados, podemos cargarlos en un DataFrame de Pandas para analizarlos. Esto nos permite tener una visión consolidada de todas las dependencias y artefactos encontrados en los diferentes repositorios.

El siguiente código se encarga de:

1.  **Localizar los archivos JSON**: Busca todos los archivos `.json` en el directorio `data/results`.
2.  **Cargar y normalizar los datos**: Para cada archivo JSON, lo carga y utiliza `pd.json_normalize` para aplanar la estructura anidada de los artefactos.
3.  **Añadir información del repositorio**: Agrega una columna `repo` para identificar a qué repositorio pertenece cada artefacto.
4.  **Concatenar los resultados**: Une todos los DataFrames individuales en uno solo.

In [2]:
import pandas as pd
from pathlib import Path
import json

path = Path("../../data/results")

df_all = pd.concat(
    [
        pd.json_normalize(json.load(open(file))["artifacts"]).assign(repo=file.stem)
        for file in path.glob("*.json")
    ],
    ignore_index=True
)

df_all.head()

,repo,id,name,version,type,foundBy,locations,licenses,language,cpes,...,metadata.dependencies.@isaacs/brace-expansion,metadata.dependencies.@yarnpkg/fslib,metadata.dependencies.chokidar,metadata.dependencies.command-line-args,metadata.dependencies.jsonc-parser,metadata.dependencies.source-map-support,metadata.dependencies.tmp,metadata.dependencies.buffer-from,metadata.dependencies.source-map,metadata.dependencies.vscode-languageclient
0,debugpy,3a316b9e62f65fbf,actions/checkout,v1,github-action,github-actions-usage-cataloger,[{'path': '/src/debugpy/_vendored/pydevd/.gith...,[],,[{'cpe': 'cpe:2.3:a:actions\/checkout:actions\...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,debugpy,494a085f2487f1b7,actions/checkout,v3,github-action,github-actions-usage-cataloger,"[{'path': '/.github/workflows/codeql.yml', 'ac...",[],,[{'cpe': 'cpe:2.3:a:actions\/checkout:actions\...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,debugpy,f57f2e09cb32520f,actions/checkout,v3,github-action,github-actions-usage-cataloger,[{'path': '/src/debugpy/_vendored/pydevd/.gith...,[],,[{'cpe': 'cpe:2.3:a:actions\/checkout:actions\...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,debugpy,70f398f17c060b79,actions/checkout,v4,github-action,github-actions-usage-cataloger,[{'path': '/src/debugpy/_vendored/pydevd/.gith...,[],,[{'cpe': 'cpe:2.3:a:actions\/checkout:actions\...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,debugpy,5b409100276c3489,actions/github-script,v6,github-action,github-actions-usage-cataloger,"[{'path': '/.github/workflows/auto-label.yml',...",[],,[{'cpe': 'cpe:2.3:a:actions\/github-script:act...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Exploración de los Resultados

Con los datos en un DataFrame, podemos realizar diversas consultas y análisis. Por ejemplo, podemos ver cuántos artefactos se encontraron por cada repositorio.

In [3]:
df_all['repo'].value_counts()

repo
semantic-kernel         1825
azure-sdk-for-python     359
pyright                   83
debugpy                   19
Name: count, dtype: int64

También podemos ver las licencias más comunes encontradas en todas las dependencias.

In [4]:
df_all['licenses'].explode().value_counts().head(10)

licenses
{'value': 'MIT', 'spdxExpression': 'MIT', 'type': 'declared', 'urls': [], 'locations': [{'path': '/dotnet/samples/Demos/ProcessFrameworkWithSignalR/src/ProcessFramework.Aspire.SignalR.ReactFrontend/package-lock.json', 'accessPath': '/dotnet/samples/Demos/ProcessFrameworkWithSignalR/src/ProcessFramework.Aspire.SignalR.ReactFrontend/package-lock.json', 'annotations': {'evidence': 'primary'}}]}    216
{'value': 'MIT', 'spdxExpression': 'MIT', 'type': 'declared', 'urls': [], 'locations': [{'path': '/dotnet/samples/Demos/ProcessWithCloudEvents/ProcessWithCloudEvents.Client/package-lock.json', 'accessPath': '/dotnet/samples/Demos/ProcessWithCloudEvents/ProcessWithCloudEvents.Client/package-lock.json', 'annotations': {'evidence': 'primary'}}]}                                                      180
{'value': 'MIT', 'spdxExpression': 'MIT', 'type': 'declared', 'urls': [], 'locations': [{'path': '/eng/common/tsp-client/package-lock.json', 'accessPath': '/eng/common/tsp-client/package-

## Cómo Añadir Nuevos Repositorios al Análisis

El proyecto está diseñado para que sea sencillo añadir nuevos repositorios al proceso de generación de SBOMs. El sistema utiliza **Git submodules** para gestionar los repositorios externos.

Sigue estos pasos para añadir un nuevo repositorio:

### 1. Edita el Archivo `repos.json`

Abre el archivo `data/repos.json`. Este archivo contiene una lista de los repositorios que se van a analizar. Para añadir uno nuevo, simplemente agrega un nuevo objeto JSON al array `repositories` con la siguiente estructura:

```json
{
  "url": "URL_DEL_REPOSITORIO_EN_GITHUB.git",
  "path": "data/repos/NOMBRE_DEL_REPOSITORIO"
}
```

- `url`: La URL `.git` del repositorio que quieres clonar.
- `path`: La ruta local donde se clonará el repositorio. Se recomienda mantenerlo dentro de `data/repos/`.

<div class="alert alert-block alert-info">
<b>Tip:</b> Si estás utilizando el **Dev Container**, este proceso está automatizado. Simplemente necesitas reconstruir el contenedor después de modificar el archivo `repos.json`, y los pasos 2 y 3 se ejecutarán automáticamente al iniciar el contenedor.</div>



### 2. Ejecuta el Script para Añadir Submódulos

Una vez que hayas guardado los cambios en `repos.json`, necesitas ejecutar el script `add_submodules.py`. Este script leerá el archivo JSON y añadirá los nuevos repositorios como submódulos de Git.

Puedes ejecutarlo desde la terminal:

```bash
uv run scripts/add_submodules.py
```

### 3. Actualiza los Submódulos

Después de añadir los submódulos, necesitas que Git los clone. Ejecuta el siguiente comando:

```bash
git submodule update --init --recursive
```



## Comparación de Repositorios (Dataset Estructurado)

Finalmente, cargamos el dataset estructurado que permite comparar los repositorios de manera consistente. Este dataset incluye métricas agregadas por repositorio.

In [ ]:
import pandas as pd
df_comparison = pd.read_csv("../../data/repo_comparison_dataset.csv")
df_comparison.sort_values(by="total_dependencies", ascending=False)